# SHAP를 활용한 모델 해석 (Explainable AI)

## 개요
- Pima Indians Diabetes 데이터셋으로 **당뇨병 발병 여부** 예측
- **SHAP (SHapley Additive exPlanations)**: 모델 예측을 설명하는 최신 기법
- **3가지 Gradient Boosting 모델** 해석: XGBoost, LightGBM, CatBoost
- 전역적 해석(Global) + 지역적 해석(Local)

## SHAP의 장점
- **게임 이론 기반**: Shapley Value로 각 특성의 기여도 정확히 계산
- **직관적 시각화**: 복잡한 모델을 쉽게 이해
- **개별 예측 설명**: 왜 이런 예측이 나왔는지 상세 분석
- **전역적 해석**: 전체 데이터에서 중요한 특성 파악
- **공정성**: 모델의 편향성 검증

## 주요 단계
1. 데이터 로드 및 모델 학습
2. SHAP 기본 개념 이해
3. 전역적 특성 중요도 분석
4. 개별 예측 해석
5. 특성 간 상호작용 분석
6. 의사결정 시각화

## 라이브러리 설치 및 임포트

In [ ]:
# 필요한 라이브러리 설치
!pip install shap catboost

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# SHAP
import shap
shap.initjs()  # JavaScript 시각화 초기화

from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# 랜덤 시드 고정
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# 한글 폰트 설정
plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

print("라이브러리 임포트 완료!")
print(f"SHAP 버전: {shap.__version__}")

## 1. 데이터 로드

**Pima Indians Diabetes 데이터셋**
- 768개 샘플, 8개 피처
- 당뇨병 발병 여부 이진 분류
- 피처명:
  - preg: 임신 횟수
  - plas: 혈당 농도
  - pres: 혈압
  - skin: 피부 두께
  - insu: 인슐린
  - mass: BMI
  - pedi: 당뇨병 가족력
  - age: 나이

In [ ]:
print("Loading Pima Indians Diabetes Dataset...")
pima = fetch_openml(name='diabetes', version=1, as_frame=True)
X = pima.data
y = pima.target.map({'tested_negative': 0, 'tested_positive': 1}).astype(int)

print(f"데이터 크기: {X.shape}")
print(f"피처 목록: {list(X.columns)}")
print(f"\n클래스 분포:\n{y.value_counts()}")
print(f"\n클래스 비율:\n{y.value_counts(normalize=True).round(3)}")

print("\n데이터 샘플:")
print(X.head())

**학습/테스트 데이터 분리**

In [ ]:
# 데이터 분할 (stratify로 클래스 비율 유지)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"\nTrain 클래스 분포:\n{y_train.value_counts()}")

# 피처명 저장 (SHAP 시각화에 사용)
feature_names = X.columns.tolist()
print(f"\n피처 개수: {len(feature_names)}개")

## 2. 모델 학습

**3가지 Gradient Boosting 모델 학습**
- XGBoost
- LightGBM
- CatBoost

각 모델의 예측을 SHAP로 해석합니다.

In [ ]:
print("\n" + "="*60)
print("모델 학습")
print("="*60)

# XGBoost
print("\n1. XGBoost 학습 중...")
xgb_model = XGBClassifier(
    n_estimators=100,
    max_depth=5,
    learning_rate=0.1,
    random_state=RANDOM_STATE,
    eval_metric='logloss',
    use_label_encoder=False
)
xgb_model.fit(X_train, y_train)
xgb_pred = xgb_model.predict(X_test)
xgb_acc = accuracy_score(y_test, xgb_pred)
print(f"   XGBoost Test Accuracy: {xgb_acc:.4f}")

# LightGBM
print("\n2. LightGBM 학습 중...")
lgbm_model = LGBMClassifier(
    n_estimators=100,
    max_depth=5,
    learning_rate=0.1,
    random_state=RANDOM_STATE,
    verbose=-1
)
lgbm_model.fit(X_train, y_train)
lgbm_pred = lgbm_model.predict(X_test)
lgbm_acc = accuracy_score(y_test, lgbm_pred)
print(f"   LightGBM Test Accuracy: {lgbm_acc:.4f}")

# CatBoost
print("\n3. CatBoost 학습 중...")
cat_model = CatBoostClassifier(
    iterations=100,
    depth=5,
    learning_rate=0.1,
    random_state=RANDOM_STATE,
    verbose=0
)
cat_model.fit(X_train, y_train)
cat_pred = cat_model.predict(X_test)
cat_acc = accuracy_score(y_test, cat_pred)
print(f"   CatBoost Test Accuracy: {cat_acc:.4f}")

print("\n 모든 모델 학습 완료!")

## 3. SHAP 기본 개념

**SHAP (SHapley Additive exPlanations)**
- 게임 이론의 Shapley Value를 기반으로 한 설명 방법
- 각 특성이 예측에 얼마나 기여했는지 정량화
- 양수 SHAP 값: 해당 클래스 예측에 긍정적 기여
- 음수 SHAP 값: 해당 클래스 예측에 부정적 기여

**SHAP Explainer 종류**
- TreeExplainer: Tree 기반 모델 (빠르고 정확)
- KernelExplainer: 모든 모델 (느리지만 범용적)
- DeepExplainer: 딥러닝 모델
- LinearExplainer: 선형 모델

In [ ]:
print("\n" + "="*60)
print("SHAP Explainer 생성")
print("="*60)

# XGBoost Explainer
print("\n1. XGBoost TreeExplainer 생성...")
explainer_xgb = shap.TreeExplainer(xgb_model)
shap_values_xgb = explainer_xgb.shap_values(X_test)
print(f"   SHAP values shape: {shap_values_xgb.shape}")

# LightGBM Explainer
print("\n2. LightGBM TreeExplainer 생성...")
explainer_lgbm = shap.TreeExplainer(lgbm_model)
shap_values_lgbm = explainer_lgbm.shap_values(X_test)
# LightGBM은 2차원 배열 반환할 수 있음 (binary classification)
if isinstance(shap_values_lgbm, list):
    shap_values_lgbm = shap_values_lgbm[1]  # Positive class
print(f"   SHAP values shape: {shap_values_lgbm.shape}")

# CatBoost Explainer
print("\n3. CatBoost TreeExplainer 생성...")
explainer_cat = shap.TreeExplainer(cat_model)
shap_values_cat = explainer_cat.shap_values(X_test)
print(f"   SHAP values shape: {shap_values_cat.shape}")

print("\n✅ 모든 Explainer 생성 완료!")

## 4. 전역적 특성 중요도 (Global Feature Importance)

**전역적 해석**
- 전체 데이터셋에서 각 특성이 얼마나 중요한지 분석
- Summary Plot: 모든 샘플의 SHAP 값 분포
- Bar Plot: 평균 절대 SHAP 값으로 중요도 순위

### 4.1 XGBoost SHAP 분석

In [ ]:
print("\n" + "="*60)
print("XGBoost - 전역적 특성 중요도")
print("="*60)

# Summary Plot (Beeswarm)
plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values_xgb, X_test, feature_names=feature_names, show=False)
plt.title('XGBoost - SHAP Summary Plot (Feature Importance)', fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

# Bar Plot (평균 절대 SHAP 값)
plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values_xgb, X_test, feature_names=feature_names, plot_type='bar', show=False)
plt.title('XGBoost - Mean Absolute SHAP Values', fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

### 4.2 LightGBM SHAP 분석

In [ ]:
print("\n" + "="*60)
print("LightGBM - 전역적 특성 중요도")
print("="*60)

# Summary Plot
plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values_lgbm, X_test, feature_names=feature_names, show=False)
plt.title('LightGBM - SHAP Summary Plot (Feature Importance)', fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

# Bar Plot
plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values_lgbm, X_test, feature_names=feature_names, plot_type='bar', show=False)
plt.title('LightGBM - Mean Absolute SHAP Values', fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

### 4.3 CatBoost SHAP 분석

In [ ]:
print("\n" + "="*60)
print("CatBoost - 전역적 특성 중요도")
print("="*60)

# Summary Plot
plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values_cat, X_test, feature_names=feature_names, show=False)
plt.title('CatBoost - SHAP Summary Plot (Feature Importance)', fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

# Bar Plot
plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values_cat, X_test, feature_names=feature_names, plot_type='bar', show=False)
plt.title('CatBoost - Mean Absolute SHAP Values', fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

### 4.4 모델별 특성 중요도 비교

In [ ]:
# 평균 절대 SHAP 값 계산
importance_xgb = np.abs(shap_values_xgb).mean(axis=0)
importance_lgbm = np.abs(shap_values_lgbm).mean(axis=0)
importance_cat = np.abs(shap_values_cat).mean(axis=0)

# DataFrame 생성
importance_df = pd.DataFrame({
    'Feature': feature_names,
    'XGBoost': importance_xgb,
    'LightGBM': importance_lgbm,
    'CatBoost': importance_cat
})

# 평균 중요도로 정렬
importance_df['Mean'] = importance_df[['XGBoost', 'LightGBM', 'CatBoost']].mean(axis=1)
importance_df = importance_df.sort_values('Mean', ascending=False)

print("\n" + "="*60)
print("모델별 특성 중요도 비교 (평균 절대 SHAP 값)")
print("="*60)
print(importance_df.to_string(index=False))

# 시각화
fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(feature_names))
width = 0.25

ax.bar(x - width, importance_df['XGBoost'], width, label='XGBoost', alpha=0.8)
ax.bar(x, importance_df['LightGBM'], width, label='LightGBM', alpha=0.8)
ax.bar(x + width, importance_df['CatBoost'], width, label='CatBoost', alpha=0.8)

ax.set_xlabel('Features', fontweight='bold')
ax.set_ylabel('Mean |SHAP value|', fontweight='bold')
ax.set_title('Feature Importance Comparison (3 Models)', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(importance_df['Feature'], rotation=45, ha='right')
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## 5. 개별 예측 해석 (Local Interpretation)

**지역적 해석**
- 특정 샘플의 예측을 상세히 분석
- Force Plot: 각 특성이 예측에 어떻게 기여했는지 시각화
- Waterfall Plot: 기본 값에서 최종 예측까지의 변화 과정

당뇨병 양성으로 예측된 샘플과 음성으로 예측된 샘플을 각각 분석합니다.

### 5.1 양성 예측 샘플 분석

In [ ]:
# 당뇨병 양성으로 예측된 샘플 찾기
positive_indices = np.where(xgb_pred == 1)[0]
if len(positive_indices) > 0:
    sample_idx = positive_indices[0]

    print("\n" + "="*60)
    print(f"양성 예측 샘플 분석 (Index: {sample_idx})")
    print("="*60)

    # 실제 레이블
    actual = y_test.iloc[sample_idx]
    predicted = xgb_pred[sample_idx]

    print(f"\n실제 레이블: {actual} ({'양성' if actual == 1 else '음성'})")
    print(f"예측 레이블: {predicted} ({'양성' if predicted == 1 else '음성'})")

    # 샘플 데이터
    print("\n샘플 데이터:")
    print(X_test.iloc[sample_idx])

    # Force Plot (정적 이미지)
    print("\nForce Plot 생성 중...")
    shap.force_plot(
        explainer_xgb.expected_value,
        shap_values_xgb[sample_idx],
        X_test.iloc[sample_idx],
        matplotlib=True,
        show=False
    )
    plt.title(f'XGBoost Force Plot - Sample {sample_idx} (Positive Prediction)',
              fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.show()

    # Waterfall Plot
    print("\nWaterfall Plot 생성 중...")
    shap.waterfall_plot(
        shap.Explanation(
            values=shap_values_xgb[sample_idx],
            base_values=explainer_xgb.expected_value,
            data=X_test.iloc[sample_idx].values,
            feature_names=feature_names
        ),
        show=False
    )
    plt.title(f'XGBoost Waterfall Plot - Sample {sample_idx}', fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.show()
else:
    print("양성으로 예측된 샘플이 없습니다.")

### 5.2 음성 예측 샘플 분석

In [ ]:
# 당뇨병 음성으로 예측된 샘플 찾기
negative_indices = np.where(xgb_pred == 0)[0]
if len(negative_indices) > 0:
    sample_idx = negative_indices[0]

    print("\n" + "="*60)
    print(f"음성 예측 샘플 분석 (Index: {sample_idx})")
    print("="*60)

    # 실제 레이블
    actual = y_test.iloc[sample_idx]
    predicted = xgb_pred[sample_idx]

    print(f"\n실제 레이블: {actual} ({'양성' if actual == 1 else '음성'})")
    print(f"예측 레이블: {predicted} ({'양성' if predicted == 1 else '음성'})")

    # 샘플 데이터
    print("\n샘플 데이터:")
    print(X_test.iloc[sample_idx])

    # Force Plot
    print("\nForce Plot 생성 중...")
    shap.force_plot(
        explainer_xgb.expected_value,
        shap_values_xgb[sample_idx],
        X_test.iloc[sample_idx],
        matplotlib=True,
        show=False
    )
    plt.title(f'XGBoost Force Plot - Sample {sample_idx} (Negative Prediction)',
              fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.show()

    # Waterfall Plot
    print("\nWaterfall Plot 생성 중...")
    shap.waterfall_plot(
        shap.Explanation(
            values=shap_values_xgb[sample_idx],
            base_values=explainer_xgb.expected_value,
            data=X_test.iloc[sample_idx].values,
            feature_names=feature_names
        ),
        show=False
    )
    plt.title(f'XGBoost Waterfall Plot - Sample {sample_idx}', fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.show()
else:
    print("음성으로 예측된 샘플이 없습니다.")

## 6. 특성별 의존성 분석 (Dependence Plot)

**Dependence Plot**
- 특정 특성의 값이 변할 때 SHAP 값이 어떻게 변하는지 분석
- 특성 간 상호작용 파악
- 비선형 관계 시각화

In [ ]:
print("\n" + "="*60)
print("주요 특성 Dependence Plot")
print("="*60)

# 상위 3개 중요 특성
top_features = importance_df.head(3)['Feature'].tolist()

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for idx, feature in enumerate(top_features):
    feature_idx = feature_names.index(feature)

    # Dependence Plot
    shap.dependence_plot(
        feature_idx,
        shap_values_xgb,
        X_test,
        feature_names=feature_names,
        ax=axes[idx],
        show=False
    )
    axes[idx].set_title(f'Dependence Plot - {feature}', fontweight='bold')

plt.suptitle('XGBoost - Top 3 Features Dependence Analysis',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 7. 특성 간 상호작용 분석

**상호작용 효과**
- 두 특성이 함께 예측에 미치는 영향
- 상호작용 강도 계산
- 비선형 패턴 발견

In [ ]:
print("\n" + "="*60)
print("특성 간 상호작용 분석")
print("="*60)

# 상위 2개 특성 간 상호작용
if len(top_features) >= 2:
    feature1 = top_features[0]
    feature2 = top_features[1]

    feature1_idx = feature_names.index(feature1)
    feature2_idx = feature_names.index(feature2)

    print(f"\n분석 대상: {feature1} vs {feature2}")

    # Dependence Plot with interaction
    plt.figure(figsize=(10, 6))
    shap.dependence_plot(
        feature1_idx,
        shap_values_xgb,
        X_test,
        feature_names=feature_names,
        interaction_index=feature2_idx,
        show=False
    )
    plt.title(f'Interaction: {feature1} vs {feature2}',
              fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

## 8. Decision Plot (의사결정 과정 시각화)

**Decision Plot**
- 여러 샘플의 의사결정 과정을 동시에 비교
- 각 특성이 순차적으로 예측에 기여하는 과정 추적
- 잘못된 예측 분석에 유용

In [ ]:
print("\n" + "="*60)
print("Decision Plot - 샘플 비교")
print("="*60)

# 양성/음성 각 5개 샘플 선택
positive_samples = np.where(y_test == 1)[0][:5]
negative_samples = np.where(y_test == 0)[0][:5]
selected_samples = np.concatenate([positive_samples, negative_samples])

# Decision Plot
plt.figure(figsize=(12, 8))
shap.decision_plot(
    explainer_xgb.expected_value,
    shap_values_xgb[selected_samples],
    X_test.iloc[selected_samples],
    feature_names=feature_names,
    show=False,
    legend_labels=['Positive (1-5)', 'Negative (6-10)'],
    legend_location='lower right'
)
plt.title('Decision Plot - Positive vs Negative Samples',
          fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 9. 종합 분석 및 인사이트

In [ ]:
print("\n" + "="*80)
print("SHAP 분석 종합 결과")
print("="*80)

print("\n📊 주요 발견사항:")
print("-" * 80)

# 1. 가장 중요한 특성 Top 3
print("\n1️⃣ 가장 중요한 특성 (3가지 모델 평균):")
for i, row in importance_df.head(3).iterrows():
    print(f"   {row['Feature']}: {row['Mean']:.4f}")

# 2. 모델 간 일치도
print("\n2️⃣ 모델 성능 비교:")
print(f"   XGBoost:  {xgb_acc:.4f}")
print(f"   LightGBM: {lgbm_acc:.4f}")
print(f"   CatBoost: {cat_acc:.4f}")

# 3. 특성별 영향 방향
print("\n3️⃣ 주요 특성의 영향:")
for feature in top_features[:3]:
    feature_idx = feature_names.index(feature)
    mean_shap = shap_values_xgb[:, feature_idx].mean()
    direction = "양성 예측" if mean_shap > 0 else "음성 예측"
    print(f"   {feature}: 평균적으로 {direction}에 기여 (mean SHAP: {mean_shap:.4f})")

print("\n\n💡 실무 활용 방안:")
print("-" * 80)
print("\n1️⃣ 의료 진단 보조:")
print("   • 의사가 AI 예측을 신뢰할 수 있도록 근거 제시")
print("   • 환자에게 당뇨병 위험 요인 설명")
print("   • 잘못된 예측의 원인 분석")

print("\n2️⃣ 모델 개선:")
print("   • 중요하지 않은 특성 제거 (모델 단순화)")
print("   • 중요한 특성의 데이터 품질 개선")
print("   • 특성 간 상호작용을 활용한 파생 변수 생성")

print("\n3️⃣ 공정성 검증:")
print("   • 나이, 성별 등 민감 속성의 과도한 영향 확인")
print("   • 편향된 예측 패턴 탐지")
print("   • 규제 준수 (설명 가능한 AI)")

print("\n4️⃣ 비즈니스 인사이트:")
print("   • 당뇨병 고위험군 특성 파악")
print("   • 예방 프로그램 타겟팅")
print("   • 건강 관리 서비스 개선")

print("\n" + "="*80)
print("✅ SHAP 분석 완료!")
print("="*80)

## 10. 추가 분석: Confusion Matrix와 SHAP

In [ ]:
# Confusion Matrix
from sklearn.metrics import confusion_matrix
import seaborn as sns

print("\n" + "="*60)
print("예측 성능 분석")
print("="*60)

# XGBoost Confusion Matrix
cm = confusion_matrix(y_test, xgb_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Negative (0)', 'Positive (1)'],
            yticklabels=['Negative (0)', 'Positive (1)'])
plt.title('XGBoost - Confusion Matrix', fontsize=14, fontweight='bold')
plt.ylabel('Actual', fontweight='bold')
plt.xlabel('Predicted', fontweight='bold')
plt.tight_layout()
plt.show()

# 분류 리포트
print("\nClassification Report:")
print(classification_report(y_test, xgb_pred,
                          target_names=['Negative (0)', 'Positive (1)']))

# 잘못 예측된 샘플 분석
misclassified = np.where(y_test.values != xgb_pred)[0]
print(f"\n잘못 예측된 샘플 수: {len(misclassified)}개 ({len(misclassified)/len(y_test)*100:.1f}%)")

if len(misclassified) > 0:
    print("\n잘못 예측된 샘플의 SHAP 값 분석:")
    misclassified_shap = np.abs(shap_values_xgb[misclassified]).mean(axis=0)

    # 정렬
    sorted_indices = np.argsort(misclassified_shap)[::-1]
    print("\n잘못된 예측에 가장 영향을 준 특성:")
    for i in range(min(5, len(sorted_indices))):
        idx = sorted_indices[i]
        print(f"   {feature_names[idx]}: {misclassified_shap[idx]:.4f}")

---
## 분석 완료

이 노트북에서는 다음을 수행했습니다:

✅ **Pima Indians Diabetes 데이터셋**으로 당뇨병 예측

✅ **3가지 Gradient Boosting 모델** (XGBoost, LightGBM, CatBoost) 학습

✅ **SHAP를 활용한 모델 해석**
   - 전역적 특성 중요도 분석
   - 개별 예측 해석 (Force Plot, Waterfall Plot)
   - 특성 간 의존성 및 상호작용 분석
   - 의사결정 과정 시각화

✅ **실무 활용 방안** 제시

### SHAP의 핵심 가치

**1. 신뢰성 (Trust)**
- 블랙박스 모델을 투명하게 만듦
- 의료진이 AI 예측을 신뢰할 수 있는 근거 제공
- 환자에게 설명 가능한 진단

**2. 디버깅 (Debugging)**
- 모델이 잘못 학습한 패턴 발견
- 데이터 품질 문제 식별
- 편향성 검증

**3. 인사이트 (Insights)**
- 도메인 지식 검증
- 새로운 패턴 발견
- 비즈니스 가치 창출

**4. 규제 준수 (Compliance)**
- GDPR, AI Act 등 설명 가능성 요구사항 충족
- 의료 AI 규제 대응
- 윤리적 AI 개발

### Pima Indians Diabetes 데이터셋 특징

**의료 데이터의 특수성**
- 작은 샘플 수 (768개)로 해석 가능성 더욱 중요
- 의료 결정에 사용되므로 설명 필수
- 민감한 개인 정보 (나이, BMI 등)
- 불균형 데이터 (Negative 65%, Positive 35%)

**SHAP 활용의 이점**
- 의사가 왜 당뇨병으로 진단되었는지 이해
- 환자에게 생활습관 개선 방향 제시
- 예방 프로그램 설계에 활용
- 모델 개선을 위한 피드백